# RealVuln Human PAG-Vul binary training

Human-authored repositories, manual-review Python findings, conflict-free source contexts.

In [ ]:
!pip install -q torch-geometric pennylane


In [ ]:
import json
import subprocess
import torch
from pathlib import Path

WORK = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')
DATASET_HANDLE = 'khangtrn2/realvuln-human-pagvul-binary-assets'

def find_input_dir() -> Path:
    def valid(path: Path) -> bool:
        return (path / 'run_config.json').is_file() and (path / 'realvuln_human_binary_graphs.pt').is_file()
    matches = sorted({path.parent for path in INPUT_ROOT.rglob('run_config.json') if valid(path.parent)})
    if not matches:
        import kagglehub
        downloaded = Path(kagglehub.dataset_download(DATASET_HANDLE))
        matches = [downloaded] if valid(downloaded) else sorted({path.parent for path in downloaded.rglob('run_config.json') if valid(path.parent)})
    if len(matches) != 1:
        mounted = sorted(path.as_posix() for path in INPUT_ROOT.iterdir())
        raise RuntimeError(f'Expected exactly one RealVuln Human input directory, found {matches}; mounted={mounted}')
    return matches[0]

def resolve_device() -> str:
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable. This notebook requires a Kaggle GPU.')
    try:
        torch.zeros(1, device='cuda').add_(1).item()
    except Exception as exc:
        raise RuntimeError(f'CUDA is unusable ({exc}). Request a T4 accelerator and rerun.') from exc
    return 'cuda'

INPUT = find_input_dir()
config = json.loads((INPUT / 'run_config.json').read_text())
attention = config['attention']
output_prefix = config.get('output_prefix', 'realvuln_human_binary')
device = resolve_device()
print(f'Running RealVuln Human PAG-Vul from {INPUT.name} on {device}')

command = [
    'python', str(INPUT / 'train_pagvul_binary.py'),
    '--dataset', str(INPUT / 'realvuln_human_binary_graphs.pt'),
    '--attention', attention,
    '--split-mode', config['split_mode'],
    '--device', device,
    '--out-dir', str(WORK / f'{output_prefix}_{attention}'),
]
for name, value in config.get('trainer_args', {}).items():
    command.extend(['--' + name.replace('_', '-'), str(value)])
subprocess.run(command, check=True)


In [ ]:
report = WORK / f'{output_prefix}_{attention}' / 'report.json'
print(report.read_text())
